# Data Cleaning

AIS receivers pick up noise alongside genuine traffic: two vessels transmitting under the same MMSI, a drifting or corrupted position, or a track that jumps farther than any real ship could travel between pings. AISdb's `aisdb.denoising_encoder` module targets these problems directly, before a track ever reaches analysis or a map. This notebook queries `example_data.db`, counts the tracks before cleaning, then runs the same segment-and-relink pipeline the GitBook page teaches and counts them again so the effect of cleaning is visible as real output, not just a claim.

**What you will learn**
- How to split tracks on time gaps with `aisdb.split_timedelta` before encoding
- How `aisdb.encode_greatcircledistance` uses `distance_threshold` and `speed_threshold` to cut and re-link segments
- How to compare track counts before and after cleaning
- How to drop anchored vessels with `aisdb.remove_pings_wrt_speed`
- Why `InlandDenoising` exists for land-locked positions and what its one-time download costs

Companion GitBook page: [Data Cleaning](https://aisviz.gitbook.io/documentation/tutorials/data-cleaning).

In [ ]:
# %pip install aisdb
# Running on Google Colab? Uncomment the line above to install AISdb before continuing.

# %pip install nest_asyncio
# nest_asyncio lets aisdb.web_interface.visualize() run its own event loop inside a notebook kernel.

## Setup

Apply `nest_asyncio` once, near the top of the notebook, so `aisdb.web_interface.visualize` can run its own event loop later without conflicting with Jupyter's. This notebook queries `example_data.db`, the bundled Gulf of Maine dataset introduced in [`1-database-loading.ipynb`](1-database-loading.ipynb), over the first week of January 2022.

In [ ]:
from datetime import datetime, timedelta

import aisdb
import nest_asyncio

nest_asyncio.apply()

dbpath = './example_data.db'

start_time = datetime.strptime('2022-01-01 00:00:00', '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime('2022-01-08 00:00:00', '%Y-%m-%d %H:%M:%S')

## Track count before cleaning

Query the week of traffic and generate tracks with `aisdb.TrackGen`, the same pattern used in the querying notebook. Counting the raw tracks first gives a baseline to compare against once the denoising pipeline runs.

In [ ]:
with aisdb.SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = aisdb.DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
    )
    rowgen = qry.gen_qry()
    raw_tracks = list(aisdb.TrackGen(rowgen, decimate=False))

print(f'{len(raw_tracks)} raw tracks before cleaning')

## Segmenting and re-linking with `split_timedelta` and `encode_greatcircledistance`

`aisdb.split_timedelta` cuts a track wherever the gap between consecutive pings exceeds `maxdelta`, which keeps `encode_greatcircledistance` from wasting effort trying to re-link pings that already belong to separate voyages. `encode_greatcircledistance` then walks each segment and cuts it further wherever the implied distance or speed between two pings is physically impossible, using `distance_threshold` (metres) and `speed_threshold` (knots). Every candidate segment is scored by `encode_score` and re-joined to its best match for the same MMSI when the score clears `minscore`, which is exactly the mechanism that separates two vessels broadcasting the same identifier back into two coherent tracks. These are the same threshold values the GitBook page uses.

In [ ]:
maxdelta = timedelta(hours=24)  # split a track wherever the time gap exceeds this
distance_threshold = 20000      # max allowed distance (metres) between consecutive pings
speed_threshold = 50            # max plausible vessel speed (knots) between consecutive pings
minscore = 1e-6                 # minimum score required to re-link two segments

with aisdb.SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = aisdb.DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
    )
    rowgen = qry.gen_qry()
    tracks = aisdb.TrackGen(rowgen, decimate=False)

    track_segments = aisdb.split_timedelta(tracks, maxdelta)
    tracks_encoded = aisdb.encode_greatcircledistance(
        track_segments,
        distance_threshold=distance_threshold,
        speed_threshold=speed_threshold,
        minscore=minscore,
    )
    encoded_tracks = list(tracks_encoded)

print(f'{len(raw_tracks)} raw tracks before cleaning')
print(f'{len(encoded_tracks)} tracks after split_timedelta + encode_greatcircledistance')

## Removing anchored vessels with `remove_pings_wrt_speed`

Vessels sitting at anchor or moored at a berth broadcast steadily over long periods, and their pings add clutter without adding trajectory information. `aisdb.remove_pings_wrt_speed` drops any ping where reported speed over ground (`sog`) is at or below `speed_threshold` knots. It only looks at reported speed, so it is cheap to run and works directly on the already-encoded tracks from the previous step.

In [ ]:
moving_tracks = list(aisdb.remove_pings_wrt_speed(iter(encoded_tracks), speed_threshold=0.5))

total_pings_before = sum(track['sog'].size for track in encoded_tracks)
total_pings_after = sum(track['sog'].size for track in moving_tracks)

print(f'{total_pings_before} pings before removing anchored vessels')
print(f'{total_pings_after} pings after removing anchored vessels (speed_threshold=0.5 knots)')

## Filtering land-locked points with `InlandDenoising`

GPS drift occasionally places a reported position on land, which `aisdb.denoising_encoder.InlandDenoising` catches by checking each point against cached land and water geometries for North America. It slots into the same generator pipeline as the other denoising steps.

```python
from aisdb.denoising_encoder import InlandDenoising

with InlandDenoising(data_dir='./inland_cache') as denoiser:
    clean_tracks = denoiser.filter_noisy_points(iter(moving_tracks))
```

This notebook does not run that cell. The first construction downloads `geo_land_water_NorthAmerica.7z` from `bigdata5.research.cs.dal.ca`, roughly 340 MB, and unpacks it into `data_dir`. That is a heavy, one-time cost that does not belong in an unattended notebook run, so treat `InlandDenoising` as the land-filter option to reach for locally when a query area includes a coastline and the download budget allows it.

## Visualizing the cleaned tracks

`aisdb.web_interface.visualize` starts a local web server and opens a browser tab, which does not work on a headless test host. Keep `SHOW_MAP` set to `False` here; flip it to `True` when running this notebook on your own machine to see the before-and-after tracks on the map.

In [ ]:
SHOW_MAP = False  # flip to True locally to open the interactive map in a browser

In [ ]:
if SHOW_MAP:
    aisdb.web_interface.visualize(
        moving_tracks,
        visualearth=True,
        open_browser=True,
    )
else:
    print('SHOW_MAP is False, skipping the interactive map.')

## Next steps

With duplicate-identifier noise separated and anchored vessels dropped, move on to [`4-data-visualization.ipynb`](4-data-visualization.ipynb) to plot the cleaned tracks, covered in more depth on the [Data Visualization GitBook page](https://aisviz.gitbook.io/documentation/tutorials/data-visualization). For the full denoising API, including `encode_score` and `InlandDenoising`, see the [Data Cleaning GitBook page](https://aisviz.gitbook.io/documentation/tutorials/data-cleaning).